# [Feature] 연속소둔로 파생변수 만들기

- 03 노트북에서 본 현상을 **모델에 넣을 숫자 컬럼(파생변수)** 으로 바꿈
- 단위: **LOT 안의 1분** (LOT 사이 공백은 메우지 않음)
- 미래 정보(다음 정비까지 남은 시간, RUL 등)는 정답(라벨)이라 안 넣음 → 데이터 누수 방지
- 데이터: KAMP 공개 제조AI데이터셋을 재구성한 교육용 재현 데이터

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
FIG_DIR = os.path.join("..", "figures")        # 그래프 저장 위치
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")          # 소둔로 온도·제어출력, 1초 주기
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")           # 강종변경 등
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")            # 부품 사전, 키 PART_CD
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")             # 정비 기록, 키 WO_NO + EQP_TAG
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")        # 부품 교체, 키 WO_NO + PART_CD

ENC = "utf-8-sig"                              # CSV는 BOM 있음

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# 그래프 한글 깨짐 방지 - 설치된 폰트 중 있는 것을 골라 rcParams에 넣습니다
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False         # 음수 기호가 □ 로 나오는 것 방지
rcParams["figure.dpi"] = 110

print("경로 확인:", os.path.exists(TEMP_CSV), os.path.exists(EVENT_CSV), os.path.exists(MAINT_CSV))

In [ ]:
temp = pd.read_csv(TEMP_CSV, encoding=ENC)
temp["MEAS_DT"] = pd.to_datetime(temp["MEAS_DT"])

event = pd.read_csv(EVENT_CSV, encoding=ENC)
event["EVT_DT"] = pd.to_datetime(event["EVT_DT"])

maint = pd.read_csv(MAINT_CSV, encoding=ENC)
maint["END_DT"] = pd.to_datetime(maint["END_DT"])

partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)
part = pd.read_csv(PART_CSV, encoding=ENC)

print("온도", temp.shape, "/ 이벤트", event.shape, "/ 정비", maint.shape, "/ 부품교체", partchg.shape, "/ 부품", part.shape)

## 1. LOT에 강종 붙이기
- 03 노트북 5-1과 같은 방법: LOT 시작 시각 = CR1 강종변경 시각 → 변경 후 강종(`AFT_VAL`)

In [ ]:
lot = temp.groupby("LOT_NO")["MEAS_DT"].agg(["min", "max"]).reset_index()
lot.columns = ["LOT_NO", "시작", "끝"]

grade = event[(event["PLANT_CD"] == "CR1") & (event["EVT_TYPE"] == "강종변경")] 
lot = lot.merge(grade[["EVT_DT", "AFT_VAL"]], left_on="시작", right_on="EVT_DT", how="left")
lot = lot.rename(columns={"AFT_VAL": "강종"}).drop(columns="EVT_DT")

temp["강종"] = temp["LOT_NO"].map(lot.set_index("LOT_NO")["강종"])
print("강종 없는 행:", temp["강종"].isna().sum())
display(lot)

## 2. 1분 단위로 줄이기
- 03에서 이상 후보로 본 채널 5개만 씀: `TC-Z3`, `TC-Z4`, `TC-Z9`, `OP-Z3`, `CP-Z4`
- LOT마다 따로 1분 평균 → LOT 사이 공백은 안 메움 (데이터 사전: 리샘플링으로 공백 메우지 말 것)

In [ ]:
COLS = ["TC-Z3", "TC-Z4", "TC-Z9", "OP-Z3", "CP-Z4"]

# 포화면 100, 아니면 0 → 1분 평균을 내면 '그 1분 중 포화였던 비율(%)'
temp["OP-Z3_포화비율"] = (temp["OP-Z3"] >= 100) * 100

pieces = []
for lot_no in lot["LOT_NO"]:                       # LOT 14개
    d = temp[temp["LOT_NO"] == lot_no].set_index("MEAS_DT")
    d_1m = d[COLS + ["OP-Z3_포화비율"]].resample("1min").mean()
    d_1m["LOT_NO"] = lot_no
    d_1m["강종"] = d["강종"].iloc[0]
    pieces.append(d_1m)

feat = pd.concat(pieces).reset_index()
print("1초", len(temp), "행 → 1분", len(feat), "행")

## 3. 파생변수

| 묶음 | 파생변수 | 계산 | 왜 |
| --- | --- | --- | --- |
| 구간 | `LOT경과_분` | LOT 시작 후 몇 분 | 강종 전환 직후 흔들림 구분 |
| 구간 | `LOT남은_분` | LOT 끝까지 몇 분 | LOT 끝 20~30분 흔들림 구분. **실시간엔 모르는 값이라 분석용, 모델 입력 X** |
| 구간 | `정지재가동` | `TC-Z4`가 1년 정상 범위(Q1-3×IQR) 아래면 1 | 정기수리 전후 정지·재가동 구간 분리 |
| 강종 보정 | `TC-Z3_강종편차`, `OP-Z3_강종편차`, `CP-Z4_강종편차` | 값 − 같은 강종 중앙값 | 강종 차이를 이상으로 오인하지 않게 |
| 흐름 | `TC-Z3·OP-Z3·CP-Z4_10분평균 / 10분표준편차` | 직전 10분 (과거만 사용) | 평소 수준과 흔들림 |
| 흐름 | `TC-Z3_변화율`, `CP-Z4_변화율` | 1분 전 대비 차이 | 급하게 변하는 순간 |
| 채널 | `OP-Z3_포화비율` | 1분 중 100%였던 비율 | 히터가 최대로 일하는 정도 |
| 채널 | `TC-Z9_30분진폭` | 직전 30분 최대 − 최소 | 주기적으로 움직이는 채널은 진폭을 봄 |
| 부품 | `가스분석계셀_교체후_일`, `가스분석계셀_수명소진율` | 직전 교체 후 경과 / 표준수명 | CP-Z4 관련 후보 부품 상태 (과거 정보만) |

### 3-1. 구간

In [ ]:
# LOT 시작·끝 시각을 각 행에 붙여서 차이 계산
lot_start = feat["LOT_NO"].map(lot.set_index("LOT_NO")["시작"])
lot_end = feat["LOT_NO"].map(lot.set_index("LOT_NO")["끝"])
feat["LOT경과_분"] = (feat["MEAS_DT"] - lot_start).dt.total_seconds() / 60
feat["LOT남은_분"] = (lot_end - feat["MEAS_DT"]).dt.total_seconds() / 60

# TC-Z4가 1년 기준 정상 범위 아래면 정지·재가동
q1 = temp["TC-Z4"].quantile(0.25)
q3 = temp["TC-Z4"].quantile(0.75)
low = q1 - 3 * (q3 - q1)
feat["정지재가동"] = (feat["TC-Z4"] < low).astype(int)

print("TC-Z4 정상 범위 하한:", round(low, 2))
stop_min = feat.groupby("LOT_NO")["정지재가동"].sum()
print("LOT별 정지재가동(분):")
print(stop_min[stop_min > 0])

### 3-2. 강종 보정

In [ ]:
# 같은 강종의 중앙값 (1초 원자료 기준)
med_tc = temp.groupby("강종")["TC-Z3"].median()
med_op = temp.groupby("강종")["OP-Z3"].median()
med_cp = temp.groupby("강종")["CP-Z4"].median()

feat["TC-Z3_강종편차"] = feat["TC-Z3"] - feat["강종"].map(med_tc)
feat["OP-Z3_강종편차"] = feat["OP-Z3"] - feat["강종"].map(med_op)
feat["CP-Z4_강종편차"] = feat["CP-Z4"] - feat["강종"].map(med_cp)

### 3-3. 흐름 (LOT 안에서만 계산)
- `rolling(10)` = 직전 10행 = 직전 10분 (1분 단위라서)
- `diff()` = 바로 전 행과의 차이 = 1분 동안 변한 양

In [ ]:
pieces = []
for lot_no in lot["LOT_NO"]:                       # LOT 14개, LOT 경계를 넘지 않게 따로 계산
    d = feat[feat["LOT_NO"] == lot_no].copy()

    d["TC-Z3_10분평균"] = d["TC-Z3"].rolling(10, min_periods=5).mean()
    d["TC-Z3_10분표준편차"] = d["TC-Z3"].rolling(10, min_periods=5).std()
    d["OP-Z3_10분평균"] = d["OP-Z3"].rolling(10, min_periods=5).mean()
    d["OP-Z3_10분표준편차"] = d["OP-Z3"].rolling(10, min_periods=5).std()
    d["CP-Z4_10분평균"] = d["CP-Z4"].rolling(10, min_periods=5).mean()
    d["CP-Z4_10분표준편차"] = d["CP-Z4"].rolling(10, min_periods=5).std()

    d["TC-Z3_변화율"] = d["TC-Z3"].diff()
    d["CP-Z4_변화율"] = d["CP-Z4"].diff()

    tc9_max = d["TC-Z9"].rolling(30, min_periods=15).max()
    tc9_min = d["TC-Z9"].rolling(30, min_periods=15).min()
    d["TC-Z9_30분진폭"] = tc9_max - tc9_min

    pieces.append(d)

feat = pd.concat(pieces)

### 3-4. 부품 (가스 분석계 셀, INS-006)
- 03 노트북 Step 1~3 순서: 연속소둔로 정비 → `WO_NO` → 부품교체 → `PART_CD`
- 각 시점에서 **가장 최근 교체 후 지난 일수** → 과거 정보만 쓰니까 누수 없음
- 첫 교체(5/19) 전 LOT는 설치일을 몰라서 NaN

In [ ]:
cal = maint[maint["EQP_TAG"] == "P1-CR1-CAL01"]
cal_chg = cal.merge(partchg, on="WO_NO")
ins = cal_chg[cal_chg["PART_CD"] == "INS-006"].sort_values("END_DT")

d1 = ins["END_DT"].iloc[0]   # 1번째 교체 완료
d2 = ins["END_DT"].iloc[1]   # 2번째
d3 = ins["END_DT"].iloc[2]   # 3번째
print("가스 분석계 셀 교체:", d1, "/", d2, "/", d3)

# 교체 시각을 순서대로 덮어쓰면 '가장 최근 교체 후' 경과일이 남음
feat["가스분석계셀_교체후_일"] = np.nan
feat.loc[feat["MEAS_DT"] >= d1, "가스분석계셀_교체후_일"] = (feat["MEAS_DT"] - d1).dt.total_seconds() / 86400
feat.loc[feat["MEAS_DT"] >= d2, "가스분석계셀_교체후_일"] = (feat["MEAS_DT"] - d2).dt.total_seconds() / 86400
feat.loc[feat["MEAS_DT"] >= d3, "가스분석계셀_교체후_일"] = (feat["MEAS_DT"] - d3).dt.total_seconds() / 86400

life_h = part.loc[part["PART_CD"] == "INS-006", "STD_LIFE_H"].iloc[0]   # 표준수명(시간)
feat["가스분석계셀_수명소진율"] = feat["가스분석계셀_교체후_일"] * 24 / life_h * 100   # %
print("표준수명:", life_h, "시간")

## 4. 결과 확인

In [ ]:
FEATURES = [
    "LOT경과_분", "정지재가동",
    "TC-Z3_강종편차", "OP-Z3_강종편차", "CP-Z4_강종편차",
    "TC-Z3_10분평균", "TC-Z3_10분표준편차",
    "OP-Z3_10분평균", "OP-Z3_10분표준편차",
    "CP-Z4_10분평균", "CP-Z4_10분표준편차",
    "TC-Z3_변화율", "CP-Z4_변화율",
    "OP-Z3_포화비율", "TC-Z9_30분진폭",
    "가스분석계셀_교체후_일", "가스분석계셀_수명소진율",
]   # 모델 입력 후보 (LOT남은_분은 분석용이라 뺌)

print("행", len(feat), "/ 파생변수", len(FEATURES), "개")
display(feat[FEATURES].describe().T.round(2))

In [ ]:
# LOT별 평균으로 한눈에 보기
display(feat.groupby(["LOT_NO", "강종"])[["정지재가동", "OP-Z3_포화비율", "CP-Z4_강종편차", "가스분석계셀_교체후_일"]].mean().round(2))

### 4-1. 파생변수가 03에서 본 현상을 잡는지 확인

**왜 이 그래프가 필요한가?**
- 03 노트북에서 LOT 240058(6월 노내 정비 전날)은 CP-Z4가 먼저 떨어지고 온도가 정지로 내려간 LOT
- 파생변수가 이걸 숫자로 제대로 잡는지 눈으로 확인

In [ ]:
d = feat[feat["LOT_NO"] == 240058]

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)

axes[0].plot(d["MEAS_DT"], d["TC-Z4"], color="#2a78d6")
axes[0].scatter(d.loc[d["정지재가동"] == 1, "MEAS_DT"], d.loc[d["정지재가동"] == 1, "TC-Z4"], color="#d03b3b", s=10, label="정지재가동 = 1")
axes[0].set_ylabel("TC-Z4")
axes[0].legend(loc="lower left")

axes[1].plot(d["MEAS_DT"], d["CP-Z4_강종편차"], color="#2a78d6")
axes[1].set_ylabel("CP-Z4_강종편차")

axes[2].plot(d["MEAS_DT"], d["CP-Z4_10분표준편차"], color="#2a78d6")
axes[2].set_ylabel("CP-Z4_10분표준편차")

axes[3].plot(d["MEAS_DT"], d["OP-Z3_포화비율"], color="#2a78d6")
axes[3].set_ylabel("OP-Z3_포화비율(%)")

axes[0].set_title("LOT 240058 (DP590) 파생변수")
plt.tight_layout()
# plt.savefig(os.path.join(FIG_DIR, "05_features_lot_240058.png"), bbox_inches="tight")
plt.show()

**읽는 법**
- `정지재가동`: LOT 끝 온도가 떨어지는 구간만 1로 잡힘
- `CP-Z4_강종편차`: 온도보다 먼저 음수로 크게 떨어짐 → 03에서 본 "CP가 먼저 움직임"이 숫자로 잡힘
- `CP-Z4_10분표준편차`: 떨어지기 시작하는 순간 튀어서 변화 시점을 잡음

In [ ]:
# 다음 노트북에서 쓰려면 저장 (data 폴더는 git에 안 올라감)
# feat.to_csv(os.path.join(DATA_DIR, "05_features_1min.csv"), index=False, encoding=ENC)

## 정리
- 파생변수 17개 (모델 입력 후보) + 분석용 1개(`LOT남은_분`)
- 한계
    - 강종당 LOT가 1~4개뿐 → SGCC·DP590·SPCD는 강종 중앙값이 그 LOT 하나에서 나옴 (강종 편차가 0 근처로 쏠림)
    - 가스 분석계 셀 교체는 3건뿐, CP-Z4와의 관계는 추정
    - 14개 LOT·약 99시간 데이터라 결과를 일반화하지 않음
- 다음: 이 파생변수로 IsolationForest 돌려서 03의 3σ·IQR 기준선과 비교